# NISAR GCOV — Efficient Large-Data Access


# 05 — Large-Data Performance & Windowed Access

A full-resolution GCOV frame won't comfortably fit in memory on a typical training laptop. This module reads a chunked window instead of loading the whole array — the same 16 GB budget assumption shows up again in Module 08.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


## Geographic context

Still tied to the Module 01 footprint.

In [ ]:
from nisar_utils.gcov import open_gcov,get_grid_coordinates
from nisar_utils.mapping import scene_extent_wgs84,plot_scene_overview,folium_scene_map
grid=f"{profile.gcov_root}/grids/{freq}"
with open_gcov(NISAR_FILE) as f: _x,_y=get_grid_coordinates(f,grid)
scene_bounds,_=scene_extent_wgs84(_x,_y,profile.epsg)
print("Scene WGS84 extent:",scene_bounds)


In [ ]:
plot_scene_overview(_x,_y,profile.epsg,title=f"NISAR {profile.sar_family} {freq} — Geographic Context")


In [ ]:
m=folium_scene_map(_x,_y,profile.epsg,title="NISAR scene — geographic context")
m


In [ ]:
from nisar_utils.gcov import open_gcov, read_window
from nisar_utils.performance import timed_read, memory_mb

grid=f"{profile.gcov_root}/grids/{freq}"
term=(diagonal_terms or terms)[0]
path=f"{grid}/{term}"

with open_gcov(NISAR_FILE) as f:
    d=f[path]
    r0=min(10000,max(0,d.shape[0]-1024))
    c0=min(15000,max(0,d.shape[1]-1024))
    a,dt=timed_read(
        read_window,f,path,r0,min(r0+1024,d.shape[0]),
        c0,min(c0+1024,d.shape[1])
    )

print("Dataset:",d.shape)
print("Term:",term)
print("Window:",a.shape)
print("Memory MB:",memory_mb(a))
print("Read seconds:",dt)
print("\nModule 05 STATUS: PASS")
